# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------- ----------------- 1.3/2.3 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 6.1 MB/s  0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   --- ------------------------------------ 1.3/15.8 MB 7.4 MB/s eta 0:00:02
   ------ --------------------------------- 2.6/15.8 MB 6.9 MB/s eta 0:00:02
   --------- ------------------------------ 3.9/15.8 MB 6.7 MB/s eta 0:00:02
   ------------- -------------------------- 5.2/15.8 MB 6.6 MB/s eta 0:00:02
   ----------------- ---------------------- 6.8/15.8 MB 6.7 MB/s eta 0:00:02
   -------------------- ------------------- 8.1/15.8 MB 6.6 MB/s eta 0:00:02
   ----------------------- ---------------- 9.4/15.8 MB 6.6 MB/s eta 0:00:01
   --------------------------- 

In [2]:
!pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached onnxruntime-1.23.2-cp310-cp310-win_amd64.whl.metadata (5.3 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.45.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached typer-0.27.3-py3-none-any.whl.metadata (16 kB)
  Using cached kubernetes-37.0.1-py2.py3-none-any.whl.metadata (1.9 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached aiohttp_retry-2.9.1-py3-none-any.whl.metadata (8.8 kB)
  Using cached coloredlogs-15.0.1-py2.py3-none-any.w

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [19]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [20]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

C:\Users\user\AppData\Local\Temp\ipykernel_5344\3815817353.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [8]:
doc_path =  "./datasets/state_of_the_union.txt"

In [9]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [21]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [22]:
from langchain_community.document_loaders import WebBaseLoader

In [23]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [24]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2209, which is longer than the specified 1000
Created a chunk of size 3190, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2313, which is longer than the specified 1000


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [25]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [26]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [27]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [28]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did President Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: President Biden described Ketanji Brown Jackson as one of the nation's top legal minds and a consensus builder. He highlighted her qualifications, mentioning her experience as a former top litigator in private practice, a former federal public defender, and her background from a family of public school educators and police officers. He noted that since her nomination, she has received a broad range of support from various organizations, including the Fraternal Order of Police and former judges appointed by both Democrats and Republicans.
Thought:I now know the final answer.  
Final Answer: President Biden described Ketanji Brown Jackson as one of the nation's top legal minds and a consensus builder, highlighting her q

{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "President Biden described Ketanji Brown Jackson as one of the nation's top legal minds and a consensus builder, highlighting her qualifications and broad support since her nomination."}

In [29]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and benefits of Ruff and Flake8 to provide a comprehensive answer about why one might choose Ruff over Flake8. 
Action: Ruff QA System
Action Input: "Why use ruff over flake8?"
Observation: Ruff can be preferred over Flake8 for several reasons:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, which is written in Python.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, while Flake8 has a smaller set. This means Ruff can catch a wider variety of linting issues.

3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.

5. **Independence from Plugins**: Ruff includes all rules natively, eliminating the need for additional plugins that F

{'input': 'Why use ruff over flake8?',
 'output': 'Ruff can be preferred over Flake8 for several reasons: it is faster due to being implemented in Rust, has a more comprehensive rule set with over 900 rules, can automatically fix lint violations, does not require custom rules or additional plugins, and offers compatibility with other tools. Additionally, its user-friendly command-line interface makes it easier to use. Overall, Ruff is seen as a more efficient and comprehensive tool for linting Python code compared to Flake8.'}

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [30]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [31]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [32]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden referred to Ketanji Brown Jackson as one of the nation's top legal minds and noted that she would continue Justice Breyer's legacy of excellence. He highlighted her background as a former top litigator in private practice, a former federal public defender, and her family history of public school educators and police officers. He also mentioned that she is a consensus builder and has received a broad range of support since her nomination, including endorsements from the Fraternal Order of Police and former judges appointed by both Democrats and Republicans.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "In the State of the Union address, Biden referred to Ketanji Brown Jackson as one of the nation's top legal minds and noted that she would continue Justice Breyer's legacy of excellence. He highlighted her background as a former top litigator in private practice, a former federal public defender, and her family history of public school educators and police officers. He also mentioned that she is a consensus builder and has received a broad range of support since her nomination, including endorsements from the Fraternal Order of Police and former judges appointed by both Democrats and Republicans."}

In [33]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to compare the features and benefits of Ruff and Flake8 to provide a comprehensive answer about why one might choose Ruff over Flake8. 
Action: Ruff QA System
Action Input: Why should someone use Ruff over Flake8?
Observation: Someone might choose to use Ruff over Flake8 for several reasons:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 900 rules, while Flake8 has a smaller set. This means Ruff can catch a wider variety of issues in the code.

3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.

5. **Independence from Plugins**: Ruff includes all its rules natively,

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 for several reasons:\n\n1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 900 rules, while Flake8 has a smaller set. This means Ruff can catch a wider variety of issues in the code.\n\n3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.\n\n4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.\n\n5. **Independence from Plugins**: Ruff includes all its rules natively, eliminating the need to manage multiple plugins as required by Flake8.\n\n6. **Compatibility**: Ruff can replace Flake8 and several of its plugins, as well as tools like Bl

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [34]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [35]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [36]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


I need to find out what tool Ruff uses to run over Jupyter Notebooks and whether the president mentioned that tool in the State of the Union address. I will first check the Ruff QA System for information about Ruff's functionality with Jupyter Notebooks.

Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought:I have found that Ruff integrates with nbQA to run over Jupyter Notebooks. Now, I need to check if the president mentioned nbQA in the State of the Union address.

Action: State of Union QA System  
Action Input: Did the president mention nbQA in the State of the Union address?  
Observation: I don't know.
Thought:I need to find out if nbQA was mentioned in the State of the Union address. Since I don't have that information yet, I will need to check again to see if there are any updates or specific mentions regarding nbQA i

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks, but it is unclear if the president mentioned nbQA or any related tools in the State of the Union address.'}

<br><br>

---

<br>

## 🚀 Your turn

Time to make this lab your own! 

Replace the `state_of_the_union.txt` dataset with something you'd actually enjoy chatting with. A great place to start is the [sonnets.txt dataset](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) —or any other .txt file from that repository. Of course, you're not limited to those options. Pick any text that interests you and see how your chatbot responds.

Here's what to do:

1. **Get your data** — Download your chosen `.txt` file into this project folder (or point `TextLoader` at it directly).
2. **Rebuild the vector store** — Load, split, and embed your new document, then create a fresh `RetrievalQA` chain for it (give it a descriptive `collection_name`!).
3. **Rewrite the tool description** — Update the `Tool`'s `name` and `description` so the agent knows *when* it should reach for this new tool instead of the Ruff or state-of-the-union ones.
4. **Rebuild the agent** — Combine your new tool with the existing Ruff tool (or drop it if you'd rather keep just your new dataset + one other source).
5. **Put it to the test** — Ask your agent:
   - A direct question that only your new dataset can answer.
   - A question that only the Ruff tool can answer.
   - A multi-hop question that requires combining *both* tools' knowledge, like the Jupyter/Ruff example above.
6. **Reflect** — In a markdown cell, briefly note whether the agent picked the right tool(s) each time, and what happened when you set `return_direct=True` vs. not.

⭐️ **Bonus points:**
- Instead of modifying this same file, create a new file `solution.ipynb` and replicate the process from scratch.

💡 **Tip:** 
- Watch the `verbose=True` agent logs closely — they show you the agent's reasoning step by step, which is the best way to understand *why* it picked a particular tool.

